In [1]:
import pickle
import mlflow
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

with open('../src/features/X_y_v2.pkl', 'rb') as f:
    X, y = pickle.load(f)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Before SMOTE - Fake: {y_train.sum()}, Real: {len(y_train)-y_train.sum()}")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"After SMOTE - Fake: {y_train_sm.sum()}, Real: {len(y_train_sm)-y_train_sm.sum()}")

c:\Users\MH Mazin\fake-job-detector\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (14304, 2008), Test: (3576, 2008)
Before SMOTE - Fake: 693, Real: 13611
After SMOTE - Fake: 13611, Real: 13611


In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fake_job_detector_v2")

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

2026/07/09 11:30:28 INFO mlflow.tracking.fluent: Experiment with name 'fake_job_detector_v2' does not exist. Creating a new experiment.


In [3]:
# 1. Logistic Regression
with mlflow.start_run(run_name="lr_v2"):
    params = {"C": 1.0, "max_iter": 1000, "class_weight": "balanced"}
    mlflow.log_params(params)
    
    lr = LogisticRegression(**params)
    lr.fit(X_train_sm, y_train_sm)
    m = evaluate(lr, X_test, y_test)
    mlflow.log_metrics(m)
    print(f"LR - F1: {m['f1']:.3f}, Recall: {m['recall']:.3f}")

LR - F1: 0.532, Recall: 0.884


In [4]:
# 2. Random Forest
with mlflow.start_run(run_name="rf_v2"):
    params = {"n_estimators": 200, "max_depth": 20, "class_weight": "balanced", "random_state": 42}
    mlflow.log_params(params)
    
    rf = RandomForestClassifier(**params)
    rf.fit(X_train_sm, y_train_sm)
    m = evaluate(rf, X_test, y_test)
    mlflow.log_metrics(m)
    print(f"RF - F1: {m['f1']:.3f}, Recall: {m['recall']:.3f}")

RF - F1: 0.807, Recall: 0.711


In [5]:
# 3. XGBoost
with mlflow.start_run(run_name="xgb_v2"):
    params = {
        "n_estimators": 200,
        "max_depth": 10,
        "learning_rate": 0.1,
        "scale_pos_weight": len(y_train[y_train==0]) / len(y_train[y_train==1]),
        "random_state": 42
    }
    mlflow.log_params(params)
    
    xgb = XGBClassifier(**params)
    xgb.fit(X_train_sm, y_train_sm)
    m = evaluate(xgb, X_test, y_test)
    mlflow.log_metrics(m)
    print(f"XGB - F1: {m['f1']:.3f}, Recall: {m['recall']:.3f}")

XGB - F1: 0.810, Recall: 0.786


In [6]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
exp = client.get_experiment_by_name("fake_job_detector_v2")
runs = client.search_runs(experiment_ids=[exp.experiment_id])

print("All runs:")
for run in runs:
    f1 = run.data.metrics.get("f1", 0)
    print(f"  {run.info.run_name}: F1={f1:.3f}")

best = max(runs, key=lambda r: r.data.metrics.get("f1", 0))
print(f"\nBest: {best.info.run_name}, F1={best.data.metrics['f1']:.3f}")

All runs:
  xgb_v2: F1=0.810
  rf_v2: F1=0.807
  lr_v2: F1=0.532

Best: xgb_v2, F1=0.810


In [7]:
# Save best model
import os
os.makedirs('../src/models', exist_ok=True)

# Determine which model is best
best_name = best.info.run_name
if 'lr' in best_name:
    best_model = lr
elif 'rf' in best_name:
    best_model = rf
else:
    best_model = xgb

with open('../src/models/best_model_v2.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../src/models/vectorizer_v2.pkl', 'wb') as f:
    with open('../src/features/tfidf_vectorizer_v2.pkl', 'rb') as v:
        pickle.dump(pickle.load(v), f)

print(f"Saved best model: {best_name}")

Saved best model: xgb_v2
